# Book Recommender Assistant Project 

## 00 - Environment Setup

### 00.1 - Install Requirements

In [ ]:
import sys, textwrap, subprocess, pkg_resources

print("Using Python:", sys.executable)
print("Python version:", sys.version)

# --- Core list of required packages ---
REQUIRED_PACKAGES = [
    # Core numerics / data
    "torch>=2.2",
    "numpy>=1.26",
    "pandas>=2.2",
    "pyarrow>=15.0",
    "fastparquet",

    # Embeddings + transformers
    "sentence-transformers>=3.0.1",
    "transformers>=4.45.0",
    "tokenizers>=0.21,<0.22",

    # ML utilities
    "scikit-learn>=1.4",
    "tqdm>=4.65",
    "requests>=2.32",
    "huggingface-hub>=0.23",

    # Notebook / misc
    "ipykernel",
    "ipython",
    "jupyter",
    "rich>=13.0",
]

print("\nInstalling / upgrading packages (this may take a few minutes)...\n")

cmd = [sys.executable, "-m", "pip", "install", "-U"] + REQUIRED_PACKAGES
print(" ".join(cmd))
subprocess.check_call(cmd)

print("\n✅ pip install finished.")

# Show versions of a few key libs
def show_version(pkg_name):
    try:
        v = pkg_resources.get_distribution(pkg_name).version
        print(f"  {pkg_name}: {v}")
    except Exception:
        print(f"  {pkg_name}: NOT FOUND")

print("\nInstalled versions:")
for name in ["torch", "numpy", "pandas", "sentence-transformers", "transformers", "tokenizers"]:
    show_version(name)

print("\nℹ️  When this finishes, it's a good idea to RESTART the kernel before continuing.")


### 00.2 - Check Imports

In [13]:
try:
    import torch
    import numpy as np
    import pandas as pd
    from sentence_transformers import SentenceTransformer
    from transformers import AutoModel, AutoTokenizer
    print("✅ Core ML imports OK")
except Exception as e:
    print("❌ Problem importing ML libs:", repr(e))

try:
    import build_desc_embeddings
    print("✅ build_desc_embeddings imported")
except Exception as e:
    print("❌ Could not import build_desc_embeddings:", repr(e))

try:
    from pseudo_user_teacher import DescEmbeddingTeacher
    print("✅ pseudo_user_teacher.DescEmbeddingTeacher imported")
except Exception as e:
    print("❌ Could not import DescEmbeddingTeacher:", repr(e))

try:
    from overarching_module import BookRecSession
    print("✅ overarching_module.BookRecSession imported")
except Exception as e:
    print("❌ Could not import BookRecSession:", repr(e))

✅ Core ML imports OK
✅ build_desc_embeddings imported
✅ pseudo_user_teacher.DescEmbeddingTeacher imported
✅ overarching_module.BookRecSession imported


### 00.3 - Configure Ollama

In [14]:
import shutil, sys, subprocess

def run_cmd(cmd):
    print("\n$", " ".join(cmd))
    try:
        out = subprocess.check_output(cmd, stderr=subprocess.STDOUT, text=True)
        print(out)
    except subprocess.CalledProcessError as e:
        print(e.output)

if shutil.which("ollama") is None:
    print("❌ 'ollama' command not found on PATH.")
    print("   Install from https://ollama.com/ and ensure `ollama` is callable in this environment.")
else:
    print("✅ Ollama found on PATH.")

    # Show version
    run_cmd(["ollama", "--version"])

    # Pull the models you use in planner / RAG
    # Adjust this list if your planner uses different models.
    MODELS = [
        "mistral",           # planner LLM
        "nomic-embed-text",  # embedding model (if used)
    ]

    for m in MODELS:
        print(f"\n=== Ensuring Ollama model '{m}' is available ===")
        run_cmd(["ollama", "pull", m])

print("\nℹ️  Make sure `ollama serve` is running in the background when you use the planner.")


✅ Ollama found on PATH.

$ ollama --version
ollama version is 0.13.1


=== Ensuring Ollama model 'mistral' is available ===

$ ollama pull mistral


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling f5074b1221da:   0% ▕                  ▏ 229 KB/4.4 GB                  pulling manifest 
pulling f5074b1221da:   0% ▕                  ▏ 7.2 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:   0% ▕                  ▏  15 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:   0% ▕                  ▏  19 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:   1% ▕                  ▏  28 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:   1% ▕                  ▏  38 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:   1% ▕                  ▏  41 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:   1% ▕                  ▏  51 MB/4.4 GB                  pulling manifes

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest 
pulling 970aa74c0a90: 100% ▕██████████████████▏ 274 MB                         
pulling c71d239df917: 100% ▕██████████████████▏  11 KB                         
pulling ce4a164fc046: 100% ▕██████████████████▏   17 B                         
pulling 31df23ea7daa: 100% ▕██████████████████▏  420 B                         
verifying sha256 digest 
writing manifest 
success 


ℹ️  Make sure `ollama serve` is running in the background when you use the planner.


## 01 - Data Preparation for the Book Assistant

In [3]:
from pathlib import Path
import sys
import importlib.util

# Assume notebook is opened from the project root
PROJECT_ROOT = Path.cwd().resolve()

print("Project root:", PROJECT_ROOT)

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROC_DIR = DATA_DIR / "processed"

print("DATA_DIR:", DATA_DIR)
print("RAW_DIR:", RAW_DIR)
print("PROC_DIR:", PROC_DIR)

# Make sure these exist
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

# Show python executable being used
print("Python executable:", sys.executable)

def load_module_from_path(path: Path, module_name: str):
    """Dynamically import a module from a given path."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Module file not found: {path}")

    spec = importlib.util.spec_from_file_location(module_name, str(path))
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module

# Load data_prep.py from data/
data_prep_path = DATA_DIR / "data_prep.py"
data_prep = load_module_from_path(data_prep_path, module_name="data_prep")

print("Loaded data_prep from:", data_prep_path)
print("data_prep has:", [name for name in dir(data_prep) if name.startswith("run_") or name.endswith("_goodreads")])


Project root: /Users/lindseyraven/Desktop/MSAI_SJSU_Work/CMPE259_NatLangProc/Project/Final_Project_Code/CMPE_259_Book_Recommender_FineTune_LLM
DATA_DIR: /Users/lindseyraven/Desktop/MSAI_SJSU_Work/CMPE259_NatLangProc/Project/Final_Project_Code/CMPE_259_Book_Recommender_FineTune_LLM/data
RAW_DIR: /Users/lindseyraven/Desktop/MSAI_SJSU_Work/CMPE259_NatLangProc/Project/Final_Project_Code/CMPE_259_Book_Recommender_FineTune_LLM/data/raw
PROC_DIR: /Users/lindseyraven/Desktop/MSAI_SJSU_Work/CMPE259_NatLangProc/Project/Final_Project_Code/CMPE_259_Book_Recommender_FineTune_LLM/data/processed
Python executable: /usr/local/bin/python3
Loaded data_prep from: /Users/lindseyraven/Desktop/MSAI_SJSU_Work/CMPE259_NatLangProc/Project/Final_Project_Code/CMPE_259_Book_Recommender_FineTune_LLM/data/data_prep.py
data_prep has: ['download_raw_goodreads', 'run_pipeline']


In [11]:
# Toggle this flag depending on whether you want to auto-download from UCSD
DOWNLOAD_RAW = True  # set to False if you've already downloaded data

data_prep.download_raw_goodreads(download=DOWNLOAD_RAW)

[download] some raw Goodreads files missing; downloading from UCSD...
[download] already exists: /Users/lindseyraven/Desktop/MSAI_SJSU_Work/CMPE259_NatLangProc/Project/Final_Project_Code/CMPE_259_Book_Recommender_FineTune_LLM/data/raw/goodreads_books.json.gz
[download] downloading goodreads_book_authors.json.gz from https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_book_authors.json.gz
[download] goodreads_book_authors.json.gz: 100.00%
[download] already exists: /Users/lindseyraven/Desktop/MSAI_SJSU_Work/CMPE259_NatLangProc/Project/Final_Project_Code/CMPE_259_Book_Recommender_FineTune_LLM/data/raw/goodreads_interactions_dedup.json.gz


In [12]:
import subprocess

prep_all_path = DATA_DIR / "prep_all.py"

if not prep_all_path.exists():
    raise FileNotFoundError(f"prep_all.py not found at {prep_all_path}")

cmd = [sys.executable, str(prep_all_path)]
print("Running:", " ".join(cmd))

result = subprocess.run(cmd)
if result.returncode != 0:
    raise RuntimeError(f"prep_all.py failed with exit code {result.returncode}")
else:
    print("prep_all.py finished successfully.")


Running: /usr/local/bin/python3 /Users/lindseyraven/Desktop/MSAI_SJSU_Work/CMPE259_NatLangProc/Project/Final_Project_Code/CMPE_259_Book_Recommender_FineTune_LLM/data/prep_all.py

=== [STEP] Base data prep (books + interactions) ===
/usr/local/bin/python3 /Users/lindseyraven/Desktop/MSAI_SJSU_Work/CMPE259_NatLangProc/Project/Final_Project_Code/CMPE_259_Book_Recommender_FineTune_LLM/data/data_prep.py
[download] download=False; skipping raw downloads.
=== data_prep.py: base data preparation ===
[books] loading from /Users/lindseyraven/Desktop/MSAI_SJSU_Work/CMPE259_NatLangProc/Project/Final_Project_Code/CMPE_259_Book_Recommender_FineTune_LLM/data/raw/goodreads_books.json.gz
[books] streaming JSON from /Users/lindseyraven/Desktop/MSAI_SJSU_Work/CMPE259_NatLangProc/Project/Final_Project_Code/CMPE_259_Book_Recommender_FineTune_LLM/data/raw/goodreads_books.json.gz
[books] first chunk shape: (100000, 29)
[books] using language column: language_code
[books] using popularity column: ratings_coun

## 02 - Generate Document Embeddings For Book Similarity 

In [ ]:
# import sys

# # Upgrade transformers + sentence-transformers together
# !{sys.executable} -m pip install -U "transformers>=4.45.0" "sentence-transformers>=3.0.0"



In [4]:
from pathlib import Path
import importlib.util

# Assuming you already computed PROJECT_ROOT earlier
BUILD_DESC_PATH = PROJECT_ROOT / "build_desc_embeddings.py"
print("build_desc_embeddings.py path:", BUILD_DESC_PATH)

def load_module_from_path(path: Path, module_name: str):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Module file not found: {path}")
    spec = importlib.util.spec_from_file_location(module_name, str(path))
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module

build_desc = load_module_from_path(BUILD_DESC_PATH, "build_desc_embeddings")

print("Loaded build_desc_embeddings. Public API:")
[name for name in dir(build_desc) if name.startswith("build_")]


build_desc_embeddings.py path: /Users/lindseyraven/Desktop/MSAI_SJSU_Work/CMPE259_NatLangProc/Project/Final_Project_Code/CMPE_259_Book_Recommender_FineTune_LLM/build_desc_embeddings.py
Loaded build_desc_embeddings. Public API:


['build_description_embeddings', 'build_text_corpus']

In [5]:
import pandas as pd
import numpy as np

embs = build_desc.build_description_embeddings(limit_books=None)
print("Embeddings shape:", embs.shape)

books_path = PROJECT_ROOT / "data" / "processed" / "books_with_genres.parquet"
emb_path = PROJECT_ROOT / "models" / "desc_embeddings.npy"

df_books = pd.read_parquet(books_path)
embs = np.load(emb_path)

print("books_with_genres shape:", df_books.shape)
print("embeddings shape:", embs.shape)

[load] Loading books from /Users/lindseyraven/Desktop/MSAI_SJSU_Work/CMPE259_NatLangProc/Project/Final_Project_Code/CMPE_259_Book_Recommender_FineTune_LLM/data/processed/books_with_genres.parquet
[info] books shape: (200000, 10)
[load] Loading authors from /Users/lindseyraven/Desktop/MSAI_SJSU_Work/CMPE259_NatLangProc/Project/Final_Project_Code/CMPE_259_Book_Recommender_FineTune_LLM/data/processed/authors.parquet
[info] authors shape: (829529, 2)
[info] built author_id→name mapping of size 829529
[authors] Resolving author IDs via authors.parquet → author_names...
[authors] Sample author_names:
    book_id                                              title author_names
0   2767052            The Hunger Games (The Hunger Games, #1)             
1         3  Harry Potter and the Sorcerer's Stone (Harry P...             
2     41865                            Twilight (Twilight, #1)             
3      2657                              To Kill a Mockingbird             
4      4671       

100%|██████████| 1563/1563 [50:50<00:00,  1.95s/it]   


[info] desc_embeddings shape: (200000, 384)
[done] Saved description embeddings to /Users/lindseyraven/Desktop/MSAI_SJSU_Work/CMPE259_NatLangProc/Project/Final_Project_Code/CMPE_259_Book_Recommender_FineTune_LLM/models/desc_embeddings.npy
Embeddings shape: (200000, 384)
books_with_genres shape: (200000, 10)
embeddings shape: (200000, 384)


## 03 - Send Prompts to LLM and Recommender Module

In [8]:
from overarching_module import BookRecSession

# Instantiate a session (loads DescEmbeddingTeacher under the hood)
session = BookRecSession()
first_msg = (
    "I absolutely loved 'The Hobbit' and 'Mistborn'. "
    "I disliked 'Twilight' and I don't want YA or heavy romance. "
    "Give me around 5 recommendations."
)

first_result = session.handle_first_turn(first_msg)

print("=== RAW PREFS FROM PLANNER ===")
import json
print(json.dumps(first_result["preferences"], indent=2))

print("\n=== RECOMMENDATIONS TO SHOW USER ===")
print(first_result["response_text"])


[init] Teacher initialized with titles, genres, authors, embeddings (genres=17, authors=66085)
[user] built user embedding from query: 47 liked-book idx (|liked_books|=2), 1 disliked-book idx (|disliked_books|=1), 0 genre-based idx (|liked_genres|=0), 0 author-based idx (|liked_authors|=0), alpha=1.0

================= FILTERED =================
      30 |
 J.R.R. Tolkien 4-Book Boxed Set: The Hobbit and The Lord of the Rings  |
 rating=4.59 |
 genres=['fantasy', 'science fiction', 'young adult', 'nonfiction', 'classics'] |
 authors=['J.R.R. Tolkien'] |
 score=0.3230 
 -----------
 1081560 |
 The History of the Hobbit, Part One: Mr. Baggins                       |
 rating=3.81 |
 genres=['fantasy', 'science fiction', 'young adult', 'nonfiction', 'classics'] |
 authors=['John D. Rateliff', 'J.R.R. Tolkien'] |
 score=0.3099 
 -----------
27415414 |
 Fool's Gold (The Dragon Lords #1)                                      |
 rating=3.85 |
 genres=['fantasy', 'science fiction'] |
 authors=['

In [9]:
feedback_msg = (
    "From that list, I liked #0 and #2, but #1 felt too slow. "
    "I'm into epic fantasy, heists, and clever magic systems. "
    "No YA, please."
)

follow_result = session.handle_followup(feedback_msg)

print("=== UPDATED PREFS FROM PLANNER ===")
import json
print(json.dumps(follow_result["preferences"], indent=2))

print("\n=== UPDATED RECOMMENDATIONS TO SHOW USER ===")
print(follow_result["response_text"])


=== FINAL FOLLOW-UP PREFS ===
{
  "liked_books": [
    {
      "title": "The Hobbit",
      "rating": 5.0
    },
    {
      "title": "Mistborn",
      "rating": 5.0
    }
  ],
  "disliked_books": [
    {
      "title": "Twilight",
      "rating": 1.0
    },
    {
      "title": "Fool's Gold (The Dragon Lords #1)",
      "rating": 2.0
    },
    {
      "title": "The Annotated Hobbit",
      "rating": 1.0
    },
    {
      "title": "Mud, Sweat and Tears",
      "rating": 1.0
    }
  ],
  "excluded_books": [
    {
      "title": "The History of the Hobbit, Part One: Mr. Baggins"
    }
  ],
  "liked_genres": [
    "epic fantasy",
    "heists",
    "clever magic systems"
  ],
  "disliked_genres": [
    "YA",
    "heavy romance"
  ],
  "liked_authors": [
    "J.R.R. Tolkien"
  ],
  "disliked_authors": [
    "J.R.R. Tolkien"
  ],
  "num_recommendations": 5
}

[user] liked_authors contributed from 249 book embeddings.
[user] built user embedding from query: 47 liked-book idx (|liked_books|=

## 04 - Final Recommender Bot Example

In [10]:
from overarching_module import BookRecSession

session = BookRecSession()

# First turn
user_msg = input("Describe your preferences (first turn):\n> ")
first_result = session.handle_first_turn(user_msg)
print("\n=== RECOMMENDATIONS ===")
print(first_result["response_text"])

while True:
    more = input("\nGive feedback about these recs (or type 'quit'):\n> ")
    if more.strip().lower() in {"quit", "exit"}:
        print("Ending session.")
        break

    follow_result = session.handle_followup(more)
    print("\n=== UPDATED RECOMMENDATIONS ===")
    print(follow_result["response_text"])


[init] Teacher initialized with titles, genres, authors, embeddings (genres=17, authors=66085)
[user] liked_genres contributed from 55184 book embeddings.
[user] built user embedding from query: 535 liked-book idx (|liked_books|=1), 0 disliked-book idx (|disliked_books|=0), 55184 genre-based idx (|liked_genres|=1), 0 author-based idx (|liked_authors|=0), alpha=1.0

================= FILTERED =================
 1053996 |
 Rediscovery  (Darkover - Chronological Order #11)                      |
 rating=3.82 |
 genres=['fantasy', 'science fiction', 'nonfiction'] |
 authors=['Marion Zimmer Bradley', 'Mercedes Lackey'] |
 score=0.7152 
 -----------
33384094 |
 Dangerous Ways (The Books of Winter, #1)                               |
 rating=4.69 |
 genres=['fantasy', 'science fiction'] |
 authors=['R.R. Virdi'] |
 score=0.7015 
 -----------
24500161 |
 Blood Red (Mundy's Landing, #1)                                        |
 rating=3.74 |
 genres=['mystery', 'horror'] |
 authors=['Wendy Cors